# 001c — Extract HeAR Embeddings + Clinical Features (v2)

特徵向量（516 維）：
- `emb_0` ~ `emb_511`：HeAR embedding（512維）
- `age`：年齡（數值，標準化）
- `gender`：性別（male=0, female=1, other=2）
- `respiratory_condition`：慢性呼吸道疾病（0/1）
- `fever_muscle_pain`：發燒或肌肉痠痛（0/1）

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import librosa
import tensorflow as tf
from huggingface_hub import from_pretrained_keras, notebook_login
from huggingface_hub.utils import HfFolder

OUTPUT_DIR   = 'data'
SAMPLE_RATE  = 16000
CLIP_SAMPLES = SAMPLE_RATE * 2

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 登入 Hugging Face

In [2]:
if HfFolder.get_token() is None:
    notebook_login()
else:
    print('已登入 Hugging Face')

已登入 Hugging Face


## 載入 HeAR

In [3]:
print('載入 HeAR 模型中...')
hear_model = from_pretrained_keras('google/hear')
serving_fn  = hear_model.signatures['serving_default']
print('HeAR 載入完成！')

載入 HeAR 模型中...


Fetching 24 files: 100%|██████████| 24/24 [00:00<?, ?it/s]

HeAR 載入完成！


## 音訊切片函式

In [4]:
def load_and_slice(file_path, sr=SAMPLE_RATE, clip_samples=CLIP_SAMPLES):
    audio, _ = librosa.load(file_path, sr=sr, mono=True)
    if len(audio) < clip_samples:
        audio = np.pad(audio, (0, clip_samples - len(audio)))
    n_clips = len(audio) // clip_samples
    return np.array([audio[i*clip_samples:(i+1)*clip_samples]
                     for i in range(n_clips)], dtype=np.float32)

def get_embedding(file_path):
    clips = load_and_slice(file_path)
    if len(clips) == 0:
        return None
    emb = serving_fn(x=tf.constant(clips))['output_0'].numpy()
    return emb.mean(axis=0)

## 提取 Embedding & 合併臨床特徵

In [5]:
df_list = pd.read_csv('data/prepared_train_coughvid_list.csv')
print(f'總樣本數: {len(df_list)}')
print(df_list['label'].value_counts())

embedding_rows = []
errors = []

for _, row in tqdm(df_list.iterrows(), total=len(df_list)):
    try:
        emb = get_embedding(row['file_path'])
        if emb is None:
            errors.append({'file': row['file_path'], 'error': 'audio too short'})
            continue

        entry = {f'emb_{i}': emb[i] for i in range(512)}

        # 四項臨床特徵
        entry['age']                   = float(row['age'])
        entry['gender_encoded']        = int(row['gender_encoded'])
        entry['respiratory_condition'] = int(row['respiratory_condition'])
        entry['fever_muscle_pain']     = int(row['fever_muscle_pain'])

        entry['filename'] = row['filename']
        entry['label']    = row['label']
        embedding_rows.append(entry)

    except Exception as e:
        errors.append({'file': row['file_path'], 'error': str(e)})

print(f'\n成功: {len(embedding_rows)} 筆，失敗: {len(errors)} 筆')

總樣本數: 1629
label
healthy        543
covid          543
symptomatic    543
Name: count, dtype: int64


100%|██████████| 1629/1629 [07:17<00:00,  3.73it/s]


成功: 1629 筆，失敗: 0 筆


## 輸出 CSV

In [6]:
EMB_COLS  = [f'emb_{i}' for i in range(512)]
CLIN_COLS = ['age', 'gender_encoded', 'respiratory_condition', 'fever_muscle_pain']
ALL_COLS  = EMB_COLS + CLIN_COLS + ['filename', 'label']

df_embeddings = pd.DataFrame(embedding_rows).reindex(columns=ALL_COLS)

out_path = os.path.join(OUTPUT_DIR, 'prepared_train_hear.csv')
df_embeddings.to_csv(out_path, index=False)

print(f'已儲存至 {out_path}')
print(f'特徵維度：512 (HeAR) + 4 (clinical) = 516 維')
print(df_embeddings['label'].value_counts())
df_embeddings[['filename','label'] + CLIN_COLS].head(10)

已儲存至 data\prepared_train_hear.csv
特徵維度：512 (HeAR) + 4 (clinical) = 516 維
label
healthy        543
covid          543
symptomatic    543
Name: count, dtype: int64


,filename,label,age,gender_encoded,respiratory_condition,fever_muscle_pain
0,e36c0dda-a5cb-4398-ae8f-235320a729e4.wav,healthy,35.0,-1,0,0
1,aug_f1b271ee-6902-4cff-b260-cfdf9bc3fa1f_pitch...,covid,46.0,0,0,1
2,7eac138a-818b-49d3-8ff8-2facdb910b71.wav,covid,35.0,1,1,1
3,fbd52f87-029f-4bd9-86ed-4743018722e8.wav,covid,40.0,1,0,0
4,96678e97-6a39-4932-9c5d-13981167bbd8.wav,covid,24.0,0,0,0
5,f028405b-2f4f-4379-ba3e-acc7a69f90be.wav,covid,25.0,1,0,0
6,0b72bbdf-fe74-4c76-8c9a-9e4a60a7b9f9.wav,symptomatic,24.0,0,0,0
7,36c3a6aa-f316-41d6-996a-be2d0a9a050b.wav,symptomatic,22.0,0,0,1
8,2bf8823d-9212-4cd9-9eb6-564cd5f1ddf4.wav,symptomatic,25.0,0,1,0
9,0733f882-d7fd-4dc5-a1b0-8aeec64fc112.wav,healthy,44.0,1,0,0


## 失敗檔案檢查（選用）

In [7]:
if errors:
    print(pd.DataFrame(errors).to_string())
else:
    print('沒有失敗的檔案！')

沒有失敗的檔案！
